In [4]:
# Parámetros
run_status = "UNKNOWN"
table_name = "PT_MSTR" # Puedes pasar esto como parámetro también si es dinámico
source_system = "SC.MFG"
target_layer = "lh_bronze_qad1"
control_table = "source_to_bronze_control"

StatementMeta(, c663200a-9eb1-4080-b241-27f647baf5dd, 6, Finished, Available, Finished, False)

In [5]:
import time
import random

# Configuración para manejo de concurrencia Delta Lake (ConcurrentAppendException)
MAX_RETRIES = 10
BASE_DELAY_SEC = 3
MAX_DELAY_SEC = 40

def is_concurrent_error(ex: Exception) -> bool:
    """Detecta si el error es por concurrencia en Delta Lake."""
    msg = str(ex).lower()
    return "concurrentappendexception" in msg or "concurrent" in msg and "delta" in msg

def update_control_with_retry():
    """Ejecuta el UPDATE con reintentos y backoff exponencial para evitar fallos por concurrencia."""
    control_table_name = f"lh_control_erp.dbo.{control_table}"
    update_condition = f"source_table = '{table_name}' AND source_system = '{source_system}' AND target_layer = '{target_layer}'"

    if run_status == "Failed":
        err_msg = "Fallo el Copy data, verfica el log para mas detalles"
    else:
        err_msg = None

    update_query = f"""
        UPDATE {control_table_name}
        SET
            last_run_status = '{run_status}',
            last_run_at = current_timestamp(),
            last_success_run_at = {'current_timestamp()' if err_msg is None else 'last_success_run_at'},
            last_message = {'NULL' if err_msg is None else f"'{err_msg}'"}
        WHERE {update_condition}
    """
    return update_query, control_table_name, update_condition

print("🏁 Proceso de logging iniciado con Spark.")

update_query, control_table_name, update_condition = update_control_with_retry()

print("Ejecutando la siguiente consulta:")
print(update_query)

last_error = None
for attempt in range(1, MAX_RETRIES + 1):
    try:
        spark.sql(update_query)
        print(f"\n✅ Log actualizado para la tabla '{table_name}' con el estado: {run_status}" + (f" (intento {attempt})" if attempt > 1 else ""))
        break
    except Exception as e:
        last_error = e
        if attempt < MAX_RETRIES and is_concurrent_error(e):
            delay = min(BASE_DELAY_SEC * (2 ** (attempt - 1)) + random.uniform(0, 1), MAX_DELAY_SEC)
            print(f"⚠️ Concurrencia detectada (intento {attempt}/{MAX_RETRIES}). Reintento en {delay:.1f}s...")
            time.sleep(delay)
        else:
            print(f"❌ Falló la actualización del log: {e}")
            raise
else:
    if last_error is not None:
        raise last_error

    

StatementMeta(, c663200a-9eb1-4080-b241-27f647baf5dd, 7, Finished, Available, Finished, False)

🏁 Proceso de logging iniciado con Spark.
Ejecutando la siguiente consulta:

        UPDATE lh_control_erp.dbo.source_to_bronze_control
        SET
            last_run_status = 'UNKNOWN',
            last_run_at = current_timestamp(),
            last_success_run_at = current_timestamp(),
            last_message = NULL
        WHERE source_table = 'PT_MSTR' AND source_system = 'SC.MFG' AND target_layer = 'lh_bronze_qad1'
    

✅ Log actualizado para la tabla 'PT_MSTR' con el estado: UNKNOWN
